# Build Constructor Standings

#### Sources
1. fact_session_results
1. dim_constructors

#### Output Columns
1. season
1. constructor id
1. constructor name
1. nationality
1. race starts
1. total points
1. number of wins
1. number of podiums
1. standing position

![](/Workspace/Users/kootinrc@gmail.com/databricks-course/formula1-project/05-analytics/formula1-gold-data-erd.png)

In [0]:
%sql
CREATE OR REPLACE VIEW formula1.gold.v_constructor_standing
AS
WITH constructor_session_summary
AS
(SELECT 
    f.season,
    c.constructor_id,
    c.constructor_name,
    c.nationality,
    COUNT(*) AS race_starts,
    sum(f.points) AS total_points,
    count_if(f.is_win) AS num_of_wins,
    count_if(f.is_podium) AS num_of_podiums
FROM formula1.gold.fact_session_results f JOIN 
formula1.gold.dim_constructors c ON f.constructor_id = c.constructor_id
GROUP BY 
f.season, 
c.constructor_id, 
c.constructor_name,
c.nationality)
SELECT season,
        constructor_id,
        constructor_name,
        nationality,
        RANK() OVER (PARTITION BY season ORDER BY total_points DESC, num_of_wins DESC) AS standing,
        race_starts,
        total_points,
        num_of_wins,
        num_of_podiums
        From constructor_session_summary;

In [0]:
%sql
SELECT * FROM formula1.gold.v_constructor_standing WHERE season = 2025